# Layerwise — Depth 8

Launches the `layerwise` experiment for every seed-triple index in
`configs/layerwise_8layer.yaml` and summarises the completed runs. Results are written to
`results/layerwise/depth_8/seed_<N>/metrics.json`.


In [ ]:
import os
import sys
from pathlib import Path

# Run from the project root regardless of the notebook's directory.
ROOT = Path.cwd()
while not (ROOT / "configs").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")


## Run

Executes the runner once per seed-triple index, skipping
any seed whose `metrics.json` already exists.

In [ ]:
import subprocess
import yaml

CONFIG = "configs/layerwise_8layer.yaml"
APPROACH = "layerwise"
results_dir = Path("results/layerwise/depth_8")

n_indices = int(yaml.safe_load(Path(CONFIG).read_text())["seeds"]["seed_triples"])
print(f"Running {APPROACH} for {n_indices} seed-triple indices ...")

for i in range(n_indices):
    if (results_dir / f"seed_{i}" / "metrics.json").exists():
        print(f"seed index {i} already complete; skipping")
        continue
    print(f"--- seed index {i}/{n_indices - 1} ---", flush=True)
    subprocess.run(
        [sys.executable, "experiments/run_layerwise.py", CONFIG, "--seed-index", str(i)],
        check=True,
    )


## Summary

In [ ]:
import json

results_dir = Path("results/layerwise/depth_8")
rows = []
for seed_dir in sorted(results_dir.glob("seed_*")):
    metrics = json.loads((seed_dir / "metrics.json").read_text())
    diag = metrics.get("training_diagnostic", {})
    rows.append((
        metrics.get("seed_index"),
        metrics["test_acc"],
        metrics["test_loss"],
        metrics.get("n_parameters"),
        diag.get("mean_param_grad_variance"),
    ))

rows.sort()
if rows:
    print(f"{len(rows)} completed runs in {results_dir}:")
    print(f"{'seed':>4}  {'test_acc':>8}  {'test_loss':>9}  "
          f"{'n_params':>8}  {'grad_var':>12}")
    for seed, acc, loss, nparams, gv in rows:
        print(f"{seed:>4}  {acc:>8.4f}  {loss:>9.4f}  "
              f"{nparams:>8}  {gv:>12.3e}")
else:
    print(f"No completed runs found in {results_dir}.")
